# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/20Aarya05/FlyrankAI_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb)

In this notebook, we map our chosen lane — **Refresh / Content Opportunity Scoring** — onto the ML loop. We define the decision context, the target proxy, the success metric, load the data to define the unit of analysis, and explain why machine learning beats a fixed heuristic rule here.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### **Chosen Lane:** Refresh / Content Opportunity Scoring
### **ML Task Type:** Ranking and Scoring (decision-support)

**Why:** 
The core business decision we want to improve is: **"Which content pages should an editor or content team review first to refresh, rewrite, or update?"**
Because editor time and capacity are finite (e.g., they can only review 20-50 pages a week), we do not need a simple binary classification ("refresh / don't refresh") as a hard decision. Instead, we need a **scoring and ranking** system that outputs a continuous priority score (0 to 100) and orders the content list. This allows the team to work down the queue starting from the highest-priority page, optimizing their impact.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**What we predict:**
We predict whether a content page is experiencing a search traffic decline (specifically, a drop in GSC impressions).

**Target definition & Source:**
The target is `is_declining_label`. It is defined as `1` when `trend_direction == 'down'` (which means GSC impressions in the last 30 days dropped by more than 20% compared to the previous 30 days), and `0` otherwise. 

**Observed Outcome vs. Defined Rule:**
This target is a proxy for an **observed outcome** in the search console data (impressions drop), not a defined product rule (like `health_score` or hand-crafted priority flags). This ensures the model learns real-world search behavior rather than copying a predefined heuristic. In the full warehouse release, we would improve this by predicting an observed outcome in a *future* calendar window relative to the feature window (e.g., predicting decline in the next 30 days using features from the prior 90 days) to prevent any leakage.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success Metric:** **Precision@K (specifically Precision@50)** and **Average Precision (AP)**.

**Why this metric:**
Since our output is a prioritized queue, we care most about the quality of the top recommendations. If an editor reviews the top 50 recommended pages, we want as many of them as possible to be true opportunities (pages that are indeed declining but have search volume/demand to justify a refresh). 

**What means 'good'?**
The base rate of declining pages in our filtered dataset is approximately **54.2%**. A random guess or simple baseline would achieve a Precision@50 close to this rate. 
- A **good** Precision@50 is **>70%** (i.e. at least 35 of the top 50 pages are true opportunities).
- The baseline rule gets **24.0%** Precision@50 because it focuses on staled pages regardless of current decline direction.
- The Random Forest model in our starter code achieves **74.0%** Precision@50, proving that learning features jointly vastly outperforms the simple rules.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
import os
import pandas as pd
import numpy as np

data_path = "../../data/raw/content_refresh_anonymized.csv"
df_raw = pd.read_csv(data_path)
print(f"Raw dataset shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

df_filtered = df_raw[(df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)].copy()
df_filtered = df_filtered.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"Filtered lane slice shape: {df_filtered.shape[0]:,} rows x {df_filtered.shape[1]} columns")

print("\nUnit of Analysis: One row represents one pseudonymized content item (page) for a specific client site,")
print("aggregated over a trailing 90-day window.")
print(f"Number of unique pages (content_ids): {df_filtered['content_id'].nunique():,}")
print(f"Number of unique clients (client_ids): {df_filtered['client_id'].nunique():,}")

Raw dataset shape: 30,000 rows x 44 columns
Filtered lane slice shape: 30,000 rows x 44 columns

Unit of Analysis: One row represents one pseudonymized content item (page) for a specific client site,
aggregated over a trailing 90-day window.
Number of unique pages (content_ids): 30,000
Number of unique clients (client_ids): 32


In [ ]:
sample_cols = [
    "content_id", "client_id", "content_age_days", "impressions_90d", 
    "clicks_90d", "ctr", "avg_position", "engagement_rate", "trend_direction"
]
df_filtered[sample_cols].head()

In [ ]:
df_filtered["is_declining_label"] = df_filtered["trend_direction"].str.lower().eq("down").astype(int)

target_counts = df_filtered["is_declining_label"].value_counts()
target_pct = df_filtered["is_declining_label"].value_counts(normalize=True) * 100

print("Target Column Distribution (is_declining_label):")
for val, count in target_counts.items():
    print(f"  Class {val} (Declining={val}): {count:,} rows ({target_pct[val]:.2f}%)")

df_filtered.groupby(["trend_direction", "is_declining_label"]).size().reset_index(name="count")

Target Column Distribution (is_declining_label):
  Class 1 (Declining=1): 16,262 rows (54.21%)
  Class 0 (Declining=0): 13,738 rows (45.79%)


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Traffic decline is driven by complex, non-linear, and interacting signals rather than a single simple cause:
1. **Position Tiers & Click-through Rates:** A CTR of 0.5% might be terrible for a page ranking in position 2 (expected CTR > 5%), but excellent for a page ranking in position 9 (expected CTR < 1%). A fixed rule like `ctr < 1.0` fails to adjust for the page's position tier.
2. **Interacting Signals:** A page might decline because it is stale (`days_since_last_update > 180`), but only if it's in a highly competitive niche (`competition == HIGH`). Alternatively, a thin page (`word_count < 1000`) might perform fine if search demand is low, but will crash under search updates if demand spikes.
3. **Client Heterogeneity:** Different clients have vastly different baseline traffic, engagement rates, and content structures. A single hard-coded threshold (e.g., `impressions > 500`) might capture all pages for a large enterprise client while filtering out all pages for a small, niche client.

An ML model (such as a Random Forest or Gradient Boosted Trees) learns these multi-dimensional decision boundaries and interaction effects automatically from data, leading to a much higher Precision@50 compared to hand-tuned heuristics.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.